# 8 · Functions and Recursion

*Functions and objects* · 9 exercises

Arguments, closures, decorators, memoization and Python's recursion limit.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-8) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-8)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import sys
from functools import lru_cache

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 8.1 · Default and keyword arguments  `Core`

**Task.** Write `power(base, exp=2)` and call it positionally, with the default, and with `exp=` as a keyword.

**Example.** `power(5)` -> `25`; `power(2, 10)` -> `1024`; `power(exp=3, base=2)` -> `8`

**Big O.** O(1) calls; the work is in `**`.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-1)

In [ ]:
# 8.1 Default and keyword arguments
def power(base, exp=2):
    raise NotImplementedError

In [ ]:
def test_ex_8_1():
    assert power(5) == 25
    assert power(2, 10) == 1024
    assert power(exp=3, base=2) == 8


test_ex_8_1()
print('8.1 passed')

### 8.2 · The mutable default trap  `Core`

**Task.** Write `add_item(item, bucket=None)` that returns a list containing the item, creating a fresh list when none is passed. Explain what `bucket=[]` would do instead.

**Example.** `add_item(1)` -> `[1]`; a second `add_item(2)` -> `[2]`, not `[1, 2]`

**Big O.** O(1).

**Note.** Default values are created once, when `def` runs. A `[]` default is shared by every call that omits the argument.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-2)

In [ ]:
# 8.2 The mutable default trap
def add_item(item, bucket=None):
    raise NotImplementedError

In [ ]:
def test_ex_8_2():
    assert add_item(1) == [1]
    assert add_item(2) == [2]
    assert add_item(3, [0]) == [0, 3]


test_ex_8_2()
print('8.2 passed')

### 8.3 · *args and **kwargs  `Core`

**Task.** Write `average(*nums)` that accepts any number of values, and `describe(**info)` returning `"key=value"` pairs sorted by key, joined with `", "`.

**Example.** `average(2, 4, 9)` -> `5.0`; `describe(b=2, a=1)` -> `"a=1, b=2"`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-3)

In [ ]:
# 8.3 *args and **kwargs
def average(*nums):
    raise NotImplementedError


def describe(**info):
    raise NotImplementedError

In [ ]:
def test_ex_8_3():
    assert average(2, 4, 9) == 5.0
    assert describe(b=2, a=1) == 'a=1, b=2'


test_ex_8_3()
print('8.3 passed')

### 8.4 · Lambdas as sort keys  `Core`

**Task.** Write `by_second(pairs)` sorting tuples by their second item, then by the first, with a lambda.

**Example.** `by_second([(1, 'b'), (2, 'a'), (0, 'b')])` -> `[(2, 'a'), (0, 'b'), (1, 'b')]`

**Big O.** O(N log N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-4)

In [ ]:
# 8.4 Lambdas as sort keys
def by_second(pairs):
    raise NotImplementedError

In [ ]:
def test_ex_8_4():
    assert by_second([(1, 'b'), (2, 'a'), (0, 'b')]) == [(2, 'a'), (0, 'b'), (1, 'b')]


test_ex_8_4()
print('8.4 passed')

### 8.5 · Closures and nonlocal  `Core`

**Task.** Write `make_counter()` returning a function that returns 1, 2, 3, ... on successive calls, keeping its count in an enclosing variable updated with `nonlocal`.

**Example.** `c = make_counter(); c(), c(), c()` -> `1, 2, 3`

**Big O.** O(1) per call.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-5)

In [ ]:
# 8.5 Closures and nonlocal
def make_counter():
    raise NotImplementedError

In [ ]:
def test_ex_8_5():
    c = make_counter()
    assert (c(), c(), c()) == (1, 2, 3)
    d = make_counter()
    assert d() == 1


test_ex_8_5()
print('8.5 passed')

### 8.6 · Write a memoizing decorator  `Practice`

**Task.** Write a decorator `memoize(fn)` that caches results in a dict keyed by the argument tuple, and use it on a recursive `ways(n)`: the number of ways to climb n stairs taking 1 or 2 steps.

**Example.** `ways(10)` -> `89`; `ways(80)` returns at once

**Big O.** O(N) with the cache, O(2^N) without.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-6)

In [ ]:
# 8.6 Write a memoizing decorator
def memoize(fn):
    raise NotImplementedError


def ways(n):
    raise NotImplementedError

In [ ]:
def test_ex_8_6():
    assert ways(10) == 89
    assert ways(80) == 37889062373143906
    calls = []

    @memoize
    def square(x):
        calls.append(x)
        return x * x
    assert [square(3), square(3), square(4)] == [9, 9, 16]
    assert calls == [3, 4]


test_ex_8_6()
print('8.6 passed')

### 8.7 · lru_cache does it for you  `Core`

**Task.** Write `grid_paths(r, c)` (moves right or down from the top-left to the bottom-right of an r by c grid) decorated with `@lru_cache(maxsize=None)`.

**Example.** `grid_paths(3, 3)` -> `6`; `grid_paths(16, 16)` -> `155117520`

**Big O.** O(R * C).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-7)

In [ ]:
# 8.7 lru_cache does it for you
@lru_cache(maxsize=None)
def grid_paths(r, c):
    raise NotImplementedError

In [ ]:
def test_ex_8_7():
    assert grid_paths(3, 3) == 6
    assert grid_paths(16, 16) == 155117520


test_ex_8_7()
print('8.7 passed')

### 8.8 · Recursion depth limits  `Core`

**Task.** Write `depth_sum(n)` recursively (n + depth_sum(n - 1)) and `depth_sum_iter(n)` with a loop. Raise the limit with `sys.setrecursionlimit(10**6)` and note that very deep recursion can still crash the interpreter.

**Example.** Both give `500500` for n = 1000; at n = 100000 prefer the loop

**Big O.** O(N) time; recursion also uses O(N) stack.

**Note.** Python's default limit is 1,000 frames. Contest solutions that recurse deeply (DFS on 10^5 nodes) should go iterative (15.8).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-8)

In [ ]:
# 8.8 Recursion depth limits
sys.setrecursionlimit(10**6)


def depth_sum(n):
    raise NotImplementedError


def depth_sum_iter(n):
    raise NotImplementedError

In [ ]:
def test_ex_8_8():
    assert depth_sum(1000) == 500500 == depth_sum_iter(1000)
    assert depth_sum_iter(100000) == 5000050000


test_ex_8_8()
print('8.8 passed')

### 8.9 · Functions as values  `Practice`

**Task.** Write `compose(f, g)` returning a function that computes f(g(x)), and `apply_n(f, n, x)` that applies f to x n times.

**Example.** `compose(str, abs)(-5)` -> `"5"`; `apply_n(lambda v: v * 2, 10, 1)` -> `1024`

**Big O.** O(N) for apply_n.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-8-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-8-9)

In [ ]:
# 8.9 Functions as values
def compose(f, g):
    raise NotImplementedError


def apply_n(f, n, x):
    raise NotImplementedError

In [ ]:
def test_ex_8_9():
    assert compose(str, abs)(-5) == '5'
    assert apply_n(lambda v: v * 2, 10, 1) == 1024


test_ex_8_9()
print('8.9 passed')